In [1]:
import os

os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] += os.pathsep + r"C:\hadoop\bin"

print("HADOOP_HOME:", os.environ["HADOOP_HOME"])

HADOOP_HOME: C:\hadoop


In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Olist_ETL")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .getOrCreate()
)

print("Spark version:", spark.version)

c:\Users\girdh\AppData\Local\Programs\Python\Python313\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark version: 4.2.0


In [3]:
import logging
from pathlib import Path

# Create logs directory
log_dir = Path("logs")
log_dir.mkdir(exist_ok=True)

# Configure logger
logger = logging.getLogger("olist_pipeline")
logger.setLevel(logging.INFO)
logger.handlers.clear()

# Create log file
file_handler = logging.FileHandler(
    log_dir / "validation.log",
    mode="a",
    encoding="utf-8"
)

# Define log format
formatter = logging.Formatter(
    "%(asctime)s | %(levelname)s | %(message)s"
)

file_handler.setFormatter(formatter)
logger.addHandler(file_handler)
logger.propagate = False

In [4]:
raw_path = r"C:\MY ALL\Analytics Study\Project\Ecommerce Data Pipeline\Data\raw"

Explicit Schema Enforcement
Implemented explicit PySpark schemas for all nine Olist datasets using StructType and StructField. Defined appropriate data types for each column, including strings, integers, doubles, and timestamps.

Replaced the previous individual CSV-loading statements with a structured loading section that reads each dataset using its predefined schema instead of relying on Spark's automatic schema inference.

This ensures consistent data types, predictable data structures, and better control over the data-loading process before validation and transformation.

In [63]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
    TimestampType
)


# 1. Customers
customers_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("customer_unique_id", StringType(), True),
    StructField("customer_zip_code_prefix", StringType(), True),
    StructField("customer_city", StringType(), True),
    StructField("customer_state", StringType(), True)
])


# 2. Orders
orders_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("order_status", StringType(), True),
    StructField("order_purchase_timestamp", TimestampType(), True),
    StructField("order_approved_at", TimestampType(), True),
    StructField("order_delivered_carrier_date", TimestampType(), True),
    StructField("order_delivered_customer_date", TimestampType(), True),
    StructField("order_estimated_delivery_date", TimestampType(), True)
])


# 3. Order Items
order_items_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("order_item_id", IntegerType(), True),
    StructField("product_id", StringType(), True),
    StructField("seller_id", StringType(), True),
    StructField("shipping_limit_date", TimestampType(), True),
    StructField("price", DoubleType(), True),
    StructField("freight_value", DoubleType(), True)
])


# 4. Payments
payments_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("payment_sequential", IntegerType(), True),
    StructField("payment_type", StringType(), True),
    StructField("payment_installments", IntegerType(), True),
    StructField("payment_value", DoubleType(), True)
])


# 5. Reviews
reviews_schema = StructType([
    StructField("review_id", StringType(), True),
    StructField("order_id", StringType(), True),
    StructField("review_score", IntegerType(), True),
    StructField("review_comment_title", StringType(), True),
    StructField("review_comment_message", StringType(), True),
    StructField("review_creation_date", TimestampType(), True),
    StructField("review_answer_timestamp", TimestampType(), True)
])


# 6. Products
products_schema = StructType([
    StructField("product_id", StringType(), True),
    StructField("product_category_name", StringType(), True),
    StructField("product_name_lenght", IntegerType(), True),
    StructField("product_description_lenght", IntegerType(), True),
    StructField("product_photos_qty", IntegerType(), True),
    StructField("product_weight_g", IntegerType(), True),
    StructField("product_length_cm", IntegerType(), True),
    StructField("product_height_cm", IntegerType(), True),
    StructField("product_width_cm", IntegerType(), True)
])


# 7. Sellers
sellers_schema = StructType([
    StructField("seller_id", StringType(), True),
    StructField("seller_zip_code_prefix", StringType(), True),
    StructField("seller_city", StringType(), True),
    StructField("seller_state", StringType(), True)
])


# 8. Category Translation
category_translation_schema = StructType([
    StructField("product_category_name", StringType(), True),
    StructField("product_category_name_english", StringType(), True)
])


# 9. Geolocation
geolocation_schema = StructType([
    StructField("geolocation_zip_code_prefix", StringType(), True),
    StructField("geolocation_lat", DoubleType(), True),
    StructField("geolocation_lng", DoubleType(), True),
    StructField("geolocation_city", StringType(), True),
    StructField("geolocation_state", StringType(), True)
])

In [64]:
def load_csv(file_path, schema, dataset_name, **read_options):
    logger.info(f"Loading dataset: {dataset_name}")

    try:
        df = (
            spark.read
            .option("header", True)
            .option("mode", "FAILFAST")
            .options(**read_options)
            .schema(schema)
            .csv(file_path)
        )

        logger.info(f"Successfully loaded dataset: {dataset_name}")
        print(f"{dataset_name}: Loading initiated successfully.")

        return df

    except Exception:
        logger.exception(f"Failed to load dataset: {dataset_name}")
        raise

In [65]:
dataset_configs = {
    "Customers": {
        "file": "olist_customers_dataset.csv",
        "schema": customers_schema
    },
    "Orders": {
        "file": "olist_orders_dataset.csv",
        "schema": orders_schema
    },
    "Order Items": {
        "file": "olist_order_items_dataset.csv",
        "schema": order_items_schema
    },
    "Payments": {
        "file": "olist_order_payments_dataset.csv",
        "schema": payments_schema
    },
    "Reviews": {
        "file": "olist_order_reviews_dataset.csv",
        "schema": reviews_schema,
        "options": {
            "multiLine": True,
            "quote": '"',
            "escape": '"'
        }
    },
    "Products": {
        "file": "olist_products_dataset.csv",
        "schema": products_schema
    },
    "Sellers": {
        "file": "olist_sellers_dataset.csv",
        "schema": sellers_schema
    },
    "Category Translation": {
        "file": "product_category_name_translation.csv",
        "schema": category_translation_schema
    },
    "Geolocation": {
        "file": "olist_geolocation_dataset.csv",
        "schema": geolocation_schema
    }
}

datasets = {}

for name, config in dataset_configs.items():
    file_path = f"{raw_path}/{config['file']}"

    datasets[name] = load_csv(
        file_path=file_path,
        schema=config["schema"],
        dataset_name=name,
        **config.get("options", {})
    )

Customers: Loading initiated successfully.
Orders: Loading initiated successfully.
Order Items: Loading initiated successfully.
Payments: Loading initiated successfully.
Reviews: Loading initiated successfully.
Products: Loading initiated successfully.
Sellers: Loading initiated successfully.
Category Translation: Loading initiated successfully.
Geolocation: Loading initiated successfully.


In [71]:
customers = datasets["Customers"]
orders = datasets["Orders"]
order_items = datasets["Order Items"]
payments = datasets["Payments"]
reviews = datasets["Reviews"]
products = datasets["Products"]
sellers = datasets["Sellers"]
category_translation = datasets["Category Translation"]
geolocation = datasets["Geolocation"]

In [69]:
for name, df in datasets.items():
    print(f"\n========== {name} ==========")
    df.printSchema()



========== Customers ==========
root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)


========== Orders ==========
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)


========== Order Items ==========
root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable =

In [68]:
from pyspark.sql import functions as F
def validate_primary_key(df, table_name, key_columns):

    # Check for NULL values
    null_condition = None

    for column in key_columns:
        condition = F.col(column).isNull()
        null_condition = condition if null_condition is None else null_condition | condition

    null_count = df.filter(null_condition).count()

    # Check for duplicate keys
    duplicate_count = (
        df.groupBy(*key_columns)
          .count()
          .filter(F.col("count") > 1)
          .count()
    )

    if null_count > 0 or duplicate_count > 0:
        raise ValueError(
            f"{table_name}: Primary key validation failed. "
            f"NULL records: {null_count}, "
            f"Duplicate keys: {duplicate_count}"
        )

    print(f"{table_name}: Primary key validation passed.")

In [70]:
# VALIDATE PRIMARY KEYS

validate_primary_key(
    customers,
    "Customers",
    ["customer_id"]
)
validate_primary_key(orders, "Orders", ["order_id"])

validate_primary_key(
    order_items, "Order Items",
    ["order_id", "order_item_id"]
)

validate_primary_key(
    payments, "Payments",
    ["order_id", "payment_sequential"]
)


validate_primary_key(products, "Products", ["product_id"])

validate_primary_key(sellers, "Sellers", ["seller_id"])

validate_primary_key(
    category_translation,
    "Category Translation",
    ["product_category_name"]
)
validate_primary_key(reviews, "Reviews", ["review_id", "order_id"])

Customers: Primary key validation passed.
Orders: Primary key validation passed.
Order Items: Primary key validation passed.
Payments: Primary key validation passed.
Products: Primary key validation passed.
Sellers: Primary key validation passed.
Category Translation: Primary key validation passed.
Reviews: Primary key validation passed.


A foreign key ensures that a value in a child table exists in the corresponding parent table.

orders.customer_id refers to customers.customer_id.
Every customer referenced by an order should exist in the Customers dataset.
left_anti returns orders whose customer_id doesn't exist in customers.


In [21]:
# 1. Orders → Customers
orphan_orders = orders.join(
    customers,
    on="customer_id",
    how="left_anti"
)

# 2. Order Items → Orders
orphan_order_items = order_items.join(
    orders,
    on="order_id",
    how="left_anti"
)

# 3. Order Items → Products
orphan_products = order_items.join(
    products,
    on="product_id",
    how="left_anti"
)

# 4. Order Items → Sellers
orphan_sellers = order_items.join(
    sellers,
    on="seller_id",
    how="left_anti"
)

# 5. Payments → Orders
orphan_payments = payments.join(
    orders,
    on="order_id",
    how="left_anti"
)

# 6. Reviews → Orders
orphan_reviews = reviews.join(
    orders,
    on="order_id",
    how="left_anti"
)

# Display validation results
fk_results = [
    ("Orders → Customers", orphan_orders),
    ("Order Items → Orders", orphan_order_items),
    ("Order Items → Products", orphan_products),
    ("Order Items → Sellers", orphan_sellers),
    ("Payments → Orders", orphan_payments),
    ("Reviews → Orders", orphan_reviews)
]

for relationship, orphan_df in fk_results:
    count = orphan_df.count()
    status = "PASSED" if count == 0 else "FAILED"

    print(
        f"{relationship}: {status} | "
        f"Orphan records: {count}"
    )

Orders → Customers: PASSED | Orphan records: 0
Order Items → Orders: PASSED | Orphan records: 0
Order Items → Products: PASSED | Orphan records: 0
Order Items → Sellers: PASSED | Orphan records: 0
Payments → Orders: PASSED | Orphan records: 0
Reviews → Orders: PASSED | Orphan records: 0


In [37]:
null_checks = {
    "Customers": (customers, ["customer_id", "customer_unique_id"]),
    "Orders": (orders, ["order_id", "customer_id", "order_status", "order_purchase_timestamp"]),
    "Order Items": (order_items, ["order_id", "product_id", "seller_id", "price", "freight_value"]),
    "Payments": (payments, ["order_id", "payment_sequential", "payment_type", "payment_value"]),
    "Reviews": (reviews, ["review_id", "order_id", "review_score"]),
    "Products": (products, ["product_id"]),
    "Sellers": (sellers, ["seller_id"]),
    "Category Translation": (category_translation, ["product_category_name"])
}

for table_name, (df, columns) in null_checks.items():

    null_condition = None

    for column in columns:
        condition = F.col(column).isNull()
        null_condition = condition if null_condition is None else null_condition | condition

    null_count = df.filter(null_condition).count()

    status = "PASSED" if null_count == 0 else "FAILED"

    print(f"{table_name}: {status} | NULL records: {null_count}")

Customers: PASSED | NULL records: 0
Orders: PASSED | NULL records: 0
Order Items: PASSED | NULL records: 0
Payments: PASSED | NULL records: 0
Reviews: PASSED | NULL records: 0
Products: PASSED | NULL records: 0
Sellers: PASSED | NULL records: 0
Category Translation: PASSED | NULL records: 0


In [36]:
value_checks = {
    "Order Items - Invalid price": (
        order_items,
        F.col("price") <= 0
    ),
    "Order Items - Invalid freight": (
        order_items,
        F.col("freight_value") < 0
    ),
    "Payments - Invalid payment value": (
        payments,
        F.col("payment_value") < 0
    ),
    "Payments - Invalid installments": (
        payments,
        F.col("payment_installments") < 0
    ),
    "Reviews - Invalid review score": (
        reviews,
        ~F.col("review_score").between(1, 5)
    ),
    "Order Items - Invalid item ID": (
        order_items,
        F.col("order_item_id") <= 0
    )
}

for check_name, (df, condition) in value_checks.items():

    invalid_count = df.filter(condition).count()

    status = "PASSED" if invalid_count == 0 else "FAILED"

    print(
        f"{check_name}: {status} | "
        f"Invalid records: {invalid_count}"
    )

Order Items - Invalid price: PASSED | Invalid records: 0
Order Items - Invalid freight: PASSED | Invalid records: 0
Payments - Invalid payment value: PASSED | Invalid records: 0
Payments - Invalid installments: PASSED | Invalid records: 0
Reviews - Invalid review score: PASSED | Invalid records: 0
Order Items - Invalid item ID: PASSED | Invalid records: 0


In [35]:
date_checks = {
    "Approval before purchase": (
        F.col("order_approved_at") < F.col("order_purchase_timestamp")
    ),
    "Carrier pickup before approval": (
        F.col("order_delivered_carrier_date") < F.col("order_approved_at")
    ),
    "Customer delivery before carrier pickup": (
        F.col("order_delivered_customer_date") < F.col("order_delivered_carrier_date")
    ),
    "Customer delivery before purchase": (
        F.col("order_delivered_customer_date") < F.col("order_purchase_timestamp")
    )
}

for check_name, condition in date_checks.items():

    invalid_count = orders.filter(condition).count()

    status = "PASSED" if invalid_count == 0 else "FAILED"

    print(
        f"{check_name}: {status} | "
        f"Invalid records: {invalid_count}"
    )

Approval before purchase: PASSED | Invalid records: 0
Carrier pickup before approval: FAILED | Invalid records: 1359
Customer delivery before carrier pickup: FAILED | Invalid records: 23
Customer delivery before purchase: PASSED | Invalid records: 0


Creation of reusable function for all the checks 

In [23]:
def validate_primary_key(df, table_name, key_columns):

    # Check for NULL values
    null_condition = None

    for column in key_columns:
        condition = F.col(column).isNull()
        null_condition = condition if null_condition is None else null_condition | condition

    null_count = df.filter(null_condition).count()

    # Check for duplicate keys
    duplicate_count = (
        df.groupBy(*key_columns)
          .count()
          .filter(F.col("count") > 1)
          .count()
    )

    if null_count > 0 or duplicate_count > 0:

        error_message = (
            f"{table_name}: Primary key validation failed. "
            f"NULL records: {null_count}, "
            f"Duplicate keys: {duplicate_count}"
        )

        logger.error(error_message)
        raise ValueError(error_message)

    success_message = f"{table_name}: Primary key validation passed."

    logger.info(success_message)
    print(success_message)

In [24]:
primary_keys = {
    "Customers": ["customer_id"],
    "Orders": ["order_id"],
    "Order Items": ["order_id", "order_item_id"],
    "Payments": ["order_id", "payment_sequential"],
    "Products": ["product_id"],
    "Sellers": ["seller_id"],
    "Category Translation": ["product_category_name"],
    "Reviews": ["review_id", "order_id"]
}

In [ ]:
#Create the foreign-key validation function
def validate_foreign_key(child_df, parent_df, child_column, parent_column, relationship):

    orphan_count = (
        child_df.select(F.col(child_column).alias("_fk"))
        .join(
            parent_df.select(F.col(parent_column).alias("_pk")),
            F.col("_fk") == F.col("_pk"),
            "left_anti"
        )
        .count()
    )

    if orphan_count > 0:
        error_message = (
            f"{relationship}: Foreign key validation failed. "
            f"Orphan records: {orphan_count}"
        )
        logger.error(error_message)
        raise ValueError(error_message)

    success_message = f"{relationship}: Foreign key validation passed."

    logger.info(success_message)
    print(success_message)

In [26]:
foreign_key_checks = [
    {
        "child_df": orders,
        "parent_df": customers,
        "child_column": "customer_id",
        "parent_column": "customer_id",
        "relationship": "Orders → Customers"
    },
    {
        "child_df": order_items,
        "parent_df": orders,
        "child_column": "order_id",
        "parent_column": "order_id",
        "relationship": "Order Items → Orders"
    },
    {
        "child_df": order_items,
        "parent_df": products,
        "child_column": "product_id",
        "parent_column": "product_id",
        "relationship": "Order Items → Products"
    },
    {
        "child_df": order_items,
        "parent_df": sellers,
        "child_column": "seller_id",
        "parent_column": "seller_id",
        "relationship": "Order Items → Sellers"
    },
    {
        "child_df": payments,
        "parent_df": orders,
        "child_column": "order_id",
        "parent_column": "order_id",
        "relationship": "Payments → Orders"
    },
    {
        "child_df": reviews,
        "parent_df": orders,
        "child_column": "order_id",
        "parent_column": "order_id",
        "relationship": "Reviews → Orders"
    }
]

In [34]:
#Reusable not null validation function

def validate_not_null(df, table_name, required_columns):

    null_condition = None

    for column in required_columns:
        condition = F.col(column).isNull()
        null_condition = (
            condition if null_condition is None
            else null_condition | condition
        )

    null_count = df.filter(null_condition).count()

    if null_count > 0:
        error_message = (
            f"{table_name}: NULL validation failed. "
            f"NULL records: {null_count}"
        )
        logger.error(error_message)
        raise ValueError(error_message)

    success_message = f"{table_name}: NULL validation passed."

    logger.info(success_message)
    print(success_message)

In [50]:
#Reusable numerical validation function

def validate_values(df, check_name, condition):

    invalid_count = df.filter(condition).count()

    if invalid_count > 0:
        error_message = (
            f"{check_name}: Value validation failed. "
            f"Invalid records: {invalid_count}"
        )
        logger.error(error_message)
        raise ValueError(error_message)

    success_message = f"{check_name}: Value validation passed."

    logger.info(success_message)
    print(success_message)
    

In [51]:
# Validate date checks function

def validate_dates(df, check_name, condition):

    invalid_count = df.filter(condition).count()

    if invalid_count > 0:
        warning_message = (
            f"{check_name} | Anomalies: {invalid_count}"
        )
        logger.warning(warning_message)
        print(f"WARNING: {warning_message}")

    else:
        success_message = f"{check_name}: Passed."

        logger.info(success_message)
        print(success_message)

The central validation function

In [52]:
tables = {
    "Customers": customers,
    "Orders": orders,
    "Order Items": order_items,
    "Payments": payments,
    "Reviews": reviews,
    "Products": products,
    "Sellers": sellers,
    "Category Translation": category_translation
}

In [53]:
def validate_all_data():

    logger.info("========== DATA VALIDATION STARTED ==========")
    print("========== DATA VALIDATION STARTED ==========")

    try:

        # 1. Primary Key Validation
        for table_name, key_columns in primary_keys.items():
            validate_primary_key(
                tables[table_name],
                table_name,
                key_columns
            )

        # 2. Foreign Key Validation
        for check in foreign_key_checks:
            validate_foreign_key(**check)

        # 3. NULL Validation
        for table_name, (df, columns) in null_checks.items():
            validate_not_null(df, table_name, columns)

        # 4. Numerical Validation
        for check_name, (df, condition) in value_checks.items():
            validate_values(df, check_name, condition)

        # 5. Date Validation
        for check_name, condition in date_checks.items():
            validate_dates(orders, check_name, condition)

        logger.info("========== DATA VALIDATION COMPLETED ==========")
        print("========== DATA VALIDATION COMPLETED ==========")

    except Exception:
        logger.exception("DATA VALIDATION FAILED")
        raise

In [72]:
validate_all_data()

========== DATA VALIDATION STARTED ==========
Customers: Primary key validation passed.
Orders: Primary key validation passed.
Order Items: Primary key validation passed.
Payments: Primary key validation passed.
Products: Primary key validation passed.
Sellers: Primary key validation passed.
Category Translation: Primary key validation passed.
Reviews: Primary key validation passed.
Orders → Customers: Foreign key validation passed.
Order Items → Orders: Foreign key validation passed.
Order Items → Products: Foreign key validation passed.
Order Items → Sellers: Foreign key validation passed.
Payments → Orders: Foreign key validation passed.
Reviews → Orders: Foreign key validation passed.
Customers: NULL validation passed.
Orders: NULL validation passed.
Order Items: NULL validation passed.
Payments: NULL validation passed.
Reviews: NULL validation passed.
Products: NULL validation passed.
Sellers: NULL validation passed.
Category Translation: NULL validation passed.
Order Items - Inva

In [55]:
log_path = log_dir / "validation.log"

print("Log file exists:", log_path.exists())

print("\nLatest log entries:\n")
print(log_path.read_text(encoding="utf-8"))

Log file exists: True

Latest log entries:

2026-11-09 20:23:16,820 | INFO | Order Items - Invalid price: Value validation passed.
2026-11-09 20:23:17,005 | INFO | Order Items - Invalid freight: Value validation passed.
2026-11-09 20:23:17,270 | INFO | Payments - Invalid payment value: Value validation passed.
2026-11-09 20:23:17,480 | INFO | Payments - Invalid installments: Value validation passed.
2026-11-09 20:23:17,812 | INFO | Reviews - Invalid review score: Value validation passed.
2026-11-09 20:23:17,999 | INFO | Order Items - Invalid item ID: Value validation passed.
2026-11-09 20:23:48,286 | INFO | Approval before purchase: Passed.
2026-11-09 20:23:48,538 | WARNING | Carrier pickup before approval | Anomalies: 1359
2026-11-09 20:23:48,738 | WARNING | Customer delivery before carrier pickup | Anomalies: 23
2026-11-09 20:23:48,924 | INFO | Customer delivery before purchase: Passed.
2026-11-09 20:24:30,856 | INFO | ========== DATA VALIDATION STARTED ==========
2026-11-09 20:24:31

In [23]:
from pyspark.sql.functions import col

reviews = (
    reviews
    .withColumn("review_creation_date", col("review_creation_date").cast("timestamp"))
    .withColumn("review_answer_timestamp", col("review_answer_timestamp").cast("timestamp"))
)

In [24]:
reviews.printSchema()

root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: integer (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: timestamp (nullable = true)
 |-- review_answer_timestamp: timestamp (nullable = true)



In [25]:
reviews = reviews.select("order_id", "review_score")

In [26]:
from pyspark.sql.functions import col

order_items = order_items.withColumn(
    "item_total",
    col("price") + col("freight_value")
)

In [27]:
order_items.select("item_total").show(5)

+------------------+
|        item_total|
+------------------+
|             72.19|
|            259.83|
|            216.87|
|             25.78|
|218.04000000000002|
+------------------+
only showing top 5 rows


In [28]:
order_items = order_items.join(
    orders,
    on="order_id",
    how="left"
)

In [29]:
order_items.show(5)

+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date|price|freight_value|        item_total|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|2017

In [30]:
order_items = order_items.join(
    customers,
    on="customer_id",
    how="left"
)

In [31]:
print(order_items.count(), len(order_items.columns))

112650 19


In [32]:
order_items = order_items.join(
    products,
    on="product_id",
    how="left"
)

In [33]:
print(order_items.count(), len(order_items.columns))

112650 27


In [34]:
order_items = order_items.join(
    sellers,
    on="seller_id",
    how="left"
)

In [35]:
print(order_items.count(), len(order_items.columns))

112650 30


In [36]:
from pyspark.sql.functions import sum

payment_summary = payments.groupBy("order_id").agg(
    sum("payment_value").alias("total_payment_value")
)

In [37]:
payment_summary.show(5)

+--------------------+-------------------+
|            order_id|total_payment_value|
+--------------------+-------------------+
|bb2d7e3141540afc2...|              37.15|
|85be7c94bcd3f908f...|              72.75|
|8ca5bdac5ebe8f2d6...| 189.07999999999998|
|54066aeaaf3ac32e7...|             148.06|
|5db54d41d5ebd6d76...|             136.26|
+--------------------+-------------------+
only showing top 5 rows


In [38]:
from pyspark.sql.functions import collect_set, concat_ws

payment_types = payments.groupBy("order_id").agg(
    concat_ws(", ", collect_set("payment_type")).alias("payment_types")
)

In [39]:
payment_types.show(5)

+--------------------+-------------+
|            order_id|payment_types|
+--------------------+-------------+
|00018f77f2f0320c5...|  credit_card|
|00042b26cf59d7ce6...|  credit_card|
|00054e8431b9d7675...|  credit_card|
|0006ec9db01a64e59...|  credit_card|
|000aed2e25dbad2f9...|  credit_card|
+--------------------+-------------+
only showing top 5 rows


In [40]:
payment_types.orderBy("order_id", ascending=False).show(5)

+--------------------+-------------+
|            order_id|payment_types|
+--------------------+-------------+
|fffe41c64501cc87c...|  credit_card|
|fffe18544ffabc95d...|  credit_card|
|fffce4705a9662cd7...|  credit_card|
|fffcd46ef2263f404...|       boleto|
|fffc94f6ce00a0058...|       boleto|
+--------------------+-------------+
only showing top 5 rows


In [41]:
from pyspark.sql.functions import max

payment_installments = payments.groupBy("order_id").agg(
    max("payment_installments").alias("max_payment_installments")
)

In [42]:
payment_installments.show(5)

+--------------------+------------------------+
|            order_id|max_payment_installments|
+--------------------+------------------------+
|bb2d7e3141540afc2...|                       1|
|85be7c94bcd3f908f...|                       1|
|8ca5bdac5ebe8f2d6...|                       2|
|54066aeaaf3ac32e7...|                       2|
|5db54d41d5ebd6d76...|                       2|
+--------------------+------------------------+
only showing top 5 rows


In [43]:
from pyspark.sql.functions import sum, collect_set, concat_ws, max, count

payment_summary = payments.groupBy("order_id").agg(
    sum("payment_value").alias("total_payment_value"),
    concat_ws(", ", collect_set("payment_type")).alias("payment_types"),
    max("payment_installments").alias("max_payment_installments"),
    count("*").alias("payment_count")
)

In [44]:
order_items = order_items.join(
    payment_summary,
    on="order_id",
    how="left"
)

In [45]:
print(order_items.count(), len(order_items.columns))

112650 34


In [46]:
reviews_raw = spark.read.csv(
    f"{raw_path}/olist_order_reviews_dataset.csv",
    header=True,
    inferSchema=False
)

reviews_raw.select(
    "order_id",
    "review_score",
    "review_comment_title",
    "review_comment_message",
    "review_creation_date",
    "review_answer_timestamp"
).show(10, truncate=False)

+--------------------------------+------------+--------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------+-----------------------+
|order_id                        |review_score|review_comment_title|review_comment_message                                                                                                                                                        |review_creation_date|review_answer_timestamp|
+--------------------------------+------------+--------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------+-----------------------+
|73fc7af87114b39712e6da79b0a377eb|4           |NULL                |NULL                                                             

In [47]:
reviews_raw.filter(
    ~col("review_score").isin("1", "2", "3", "4", "5")
).select(
    "order_id",
    "review_score"
).show(20, truncate=False)

+-------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------+
|order_id                                                                                                     |review_score                                                     |
+-------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------+
|2018-03-23 00:00:00                                                                                          |2018-04-01 00:27:51                                              |
|2018-01-28 00:00:00                                                                                          |2018-02-04 11:06:15                                              |
|2017-06-24 00:00:00                                                                                          

In [48]:
reviews_raw = spark.read.csv(
    f"{raw_path}/olist_order_reviews_dataset.csv",
    header=True,
    inferSchema=False,
    multiLine=True,
    quote='"',
    escape='"'
)

reviews_raw.select(
    "order_id",
    "review_score",
    "review_creation_date",
    "review_answer_timestamp"
).show(10, truncate=False)

+--------------------------------+------------+--------------------+-----------------------+
|order_id                        |review_score|review_creation_date|review_answer_timestamp|
+--------------------------------+------------+--------------------+-----------------------+
|73fc7af87114b39712e6da79b0a377eb|4           |2018-01-18 00:00:00 |2018-01-18 21:46:59    |
|a548910a1c6147796b98fdf73dbeba33|5           |2018-03-10 00:00:00 |2018-03-11 03:05:13    |
|f9e4b658b201a9f2ecdecbb34bed034b|5           |2018-02-17 00:00:00 |2018-02-18 14:36:24    |
|658677c97b385a9be170737859d3511b|5           |2017-04-21 00:00:00 |2017-04-21 22:02:06    |
|8e6bfb81e283fa7e4f11123a3fb894f1|5           |2018-03-01 00:00:00 |2018-03-02 10:26:53    |
|b18dcdf73be66366873cd26c5724d1dc|1           |2018-04-13 00:00:00 |2018-04-16 00:39:37    |
|e48aa0d2dcec3a2e87348811bcfdf22b|5           |2017-07-16 00:00:00 |2017-07-18 19:30:34    |
|c31a859e34e3adac22f376954e19b39d|5           |2018-08-14 00:00:00 |20

In [49]:
reviews = reviews_raw.select(
    "order_id",
    col("review_score").cast("integer").alias("review_score")
)

reviews.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- review_score: integer (nullable = true)



In [50]:

from pyspark.sql.functions import count, avg, min, max
review_summary = reviews.groupBy("order_id").agg(
    count("*").alias("review_count"),
    avg("review_score").alias("review_score_avg"),
    min("review_score").alias("review_score_min"),
    max("review_score").alias("review_score_max")
)

review_summary.show(5)

+--------------------+------------+----------------+----------------+----------------+
|            order_id|review_count|review_score_avg|review_score_min|review_score_max|
+--------------------+------------+----------------+----------------+----------------+
|e239d280236cdd3c4...|           1|             5.0|               5|               5|
|d451da9b109e1786f...|           1|             5.0|               5|               5|
|0de5dbc9f32267616...|           1|             4.0|               4|               4|
|251f0a3981c4a8cb8...|           1|             5.0|               5|               5|
|f63a31c3349b87273...|           2|             5.0|               5|               5|
+--------------------+------------+----------------+----------------+----------------+
only showing top 5 rows


In [51]:
order_items = order_items.join(
    review_summary,
    on="order_id",
    how="left"
)

print(order_items.count(), len(order_items.columns))

112650 38


In [52]:
order_items = order_items.join(
    category_translation,
    on="product_category_name",
    how="left"
)

print(order_items.count(), len(order_items.columns))

112650 39


In [53]:
order_summary = order_items.groupBy("order_id").agg(
    count("*").alias("order_item_count"),
    sum("item_total").alias("order_total_value"),
    sum("freight_value").alias("order_total_freight")
)

In [54]:
order_summary.show(5)

+--------------------+----------------+------------------+-------------------+
|            order_id|order_item_count| order_total_value|order_total_freight|
+--------------------+----------------+------------------+-------------------+
|014405982914c2cde...|               2| 78.42999999999999|               29.2|
|019886de8f385a39b...|               1|             188.4|               28.5|
|01a6ad782455876aa...|               1|             50.09|               15.1|
|01d907b3e209269e1...|               1|169.76000000000002|              17.77|
|028dc52e12ddda803...|               1|             61.72|              11.73|
+--------------------+----------------+------------------+-------------------+
only showing top 5 rows


In [55]:
order_items = order_items.join(
    order_summary,
    on="order_id",
    how="left"
)

print(order_items.count(), len(order_items.columns))

112650 42


In [56]:
from pyspark.sql.functions import datediff

order_items = order_items.withColumn(
    "delivery_days",
    datediff("order_delivered_customer_date", 
            "order_purchase_timestamp")
)

In [57]:
order_items.select("order_id", "delivery_days").show(5)

+--------------------+-------------+
|            order_id|delivery_days|
+--------------------+-------------+
|00010242fe8c5a6d1...|            7|
|00018f77f2f0320c5...|           16|
|000229ec398224ef6...|            8|
|00024acbcdf0a6daa...|            6|
|00042b26cf59d7ce6...|           25|
+--------------------+-------------+
only showing top 5 rows


In [58]:
order_items = order_items.withColumn(
    "estimated_delivery_days",
    datediff(
        "order_estimated_delivery_date",
        "order_purchase_timestamp"
    )
)

order_items.select(
    "order_id",
    "estimated_delivery_days"
).show(5)

+--------------------+-----------------------+
|            order_id|estimated_delivery_days|
+--------------------+-----------------------+
|00010242fe8c5a6d1...|                     16|
|00018f77f2f0320c5...|                     19|
|000229ec398224ef6...|                     22|
|00024acbcdf0a6daa...|                     12|
|00042b26cf59d7ce6...|                     41|
+--------------------+-----------------------+
only showing top 5 rows


In [59]:
order_items = order_items.withColumn(
    "delivery_delay_days",
    datediff(
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    )
)

order_items.select(
    "order_id",
    "delivery_delay_days"
).show(10)

+--------------------+-------------------+
|            order_id|delivery_delay_days|
+--------------------+-------------------+
|00010242fe8c5a6d1...|                 -9|
|00018f77f2f0320c5...|                 -3|
|000229ec398224ef6...|                -14|
|00024acbcdf0a6daa...|                 -6|
|00042b26cf59d7ce6...|                -16|
|00048cc3ae777c65d...|                -15|
|00054e8431b9d7675...|                -17|
|000576fe39319847c...|                -16|
|0005a1a1728c9d785...|                  0|
|0005f50442cb953dc...|                -19|
+--------------------+-------------------+
only showing top 10 rows


In [60]:
from pyspark.sql.functions import when

order_items = order_items.withColumn(
    "is_delivered_late",
    when(col("delivery_delay_days") > 0, 1).otherwise(0)
)

order_items.select(
    "order_id",
    "delivery_delay_days",
    "is_delivered_late"
).show(10)

+--------------------+-------------------+-----------------+
|            order_id|delivery_delay_days|is_delivered_late|
+--------------------+-------------------+-----------------+
|00010242fe8c5a6d1...|                 -9|                0|
|00018f77f2f0320c5...|                 -3|                0|
|000229ec398224ef6...|                -14|                0|
|00024acbcdf0a6daa...|                 -6|                0|
|00042b26cf59d7ce6...|                -16|                0|
|00048cc3ae777c65d...|                -15|                0|
|00054e8431b9d7675...|                -17|                0|
|000576fe39319847c...|                -16|                0|
|0005a1a1728c9d785...|                  0|                0|
|0005f50442cb953dc...|                -19|                0|
+--------------------+-------------------+-----------------+
only showing top 10 rows


In [61]:
order_items = order_items.withColumn(
    "delivery_status",
    when(col("delivery_delay_days") > 0, "Late")
    .when(col("delivery_delay_days") == 0, "On Time")
    .otherwise("Early")
)

order_items.select(
    "order_id",
    "delivery_delay_days",
    "delivery_status"
).show(10)

+--------------------+-------------------+---------------+
|            order_id|delivery_delay_days|delivery_status|
+--------------------+-------------------+---------------+
|00010242fe8c5a6d1...|                 -9|          Early|
|00018f77f2f0320c5...|                 -3|          Early|
|000229ec398224ef6...|                -14|          Early|
|00024acbcdf0a6daa...|                 -6|          Early|
|00042b26cf59d7ce6...|                -16|          Early|
|00048cc3ae777c65d...|                -15|          Early|
|00054e8431b9d7675...|                -17|          Early|
|000576fe39319847c...|                -16|          Early|
|0005a1a1728c9d785...|                  0|        On Time|
|0005f50442cb953dc...|                -19|          Early|
+--------------------+-------------------+---------------+
only showing top 10 rows


In [62]:
order_items = order_items.withColumn(
    "delivery_status",
    when(col("delivery_delay_days").isNull(), "Not Delivered")
    .when(col("delivery_delay_days") > 0, "Late")
    .when(col("delivery_delay_days") == 0, "On Time")
    .otherwise("Early")
)

order_items.select(
    "order_id",
    "delivery_delay_days",
    "delivery_status"
).show(10)

+--------------------+-------------------+---------------+
|            order_id|delivery_delay_days|delivery_status|
+--------------------+-------------------+---------------+
|00010242fe8c5a6d1...|                 -9|          Early|
|00018f77f2f0320c5...|                 -3|          Early|
|000229ec398224ef6...|                -14|          Early|
|00024acbcdf0a6daa...|                 -6|          Early|
|00042b26cf59d7ce6...|                -16|          Early|
|00048cc3ae777c65d...|                -15|          Early|
|00054e8431b9d7675...|                -17|          Early|
|000576fe39319847c...|                -16|          Early|
|0005a1a1728c9d785...|                  0|        On Time|
|0005f50442cb953dc...|                -19|          Early|
+--------------------+-------------------+---------------+
only showing top 10 rows


In [63]:
customer_order_summary = order_items.groupBy("customer_id").agg(
    count("order_id").alias("customer_order_count")
)

customer_order_summary.show(10)

+--------------------+--------------------+
|         customer_id|customer_order_count|
+--------------------+--------------------+
|2e45292979b9b2700...|                   3|
|fc021f2dd664b4945...|                   1|
|03a7750fc7a7bfbd7...|                   1|
|456c1e01c8ed3b83a...|                   1|
|25ed0dc24f7c50b87...|                   2|
|6b994d4f438ab8322...|                   1|
|adf1ecaca5e192f13...|                   1|
|43d9a8df81f815755...|                   1|
|a7ab31829dc9a10f3...|                   6|
|98f405f18fec61e1a...|                   1|
+--------------------+--------------------+
only showing top 10 rows


In [64]:
from pyspark.sql.functions import countDistinct

customer_order_summary = order_items.groupBy("customer_id").agg(
    countDistinct("order_id").alias("customer_order_count")
)

customer_order_summary.show(10)

+--------------------+--------------------+
|         customer_id|customer_order_count|
+--------------------+--------------------+
|088935be5a63fd416...|                   1|
|1187acd131fe97cfc...|                   1|
|0b60138b038f460ba...|                   1|
|6b75e2df886b0fdde...|                   1|
|fdd48a32f403170cf...|                   1|
|42da09831872a4ecc...|                   1|
|c920352ccf11e554a...|                   1|
|6442504c76c94895e...|                   1|
|e139948a02a65f3d6...|                   1|
|bb36d8c5b34d858d1...|                   1|
+--------------------+--------------------+
only showing top 10 rows


In [65]:
customer_order_values = order_items.select(
    "customer_id",
    "order_id",
    "order_total_value"
).dropDuplicates(["customer_id", "order_id"])

customer_spend_summary = customer_order_values.groupBy("customer_id").agg(
    sum("order_total_value").alias("customer_total_spend")
)

customer_spend_summary.show(10)

+--------------------+--------------------+
|         customer_id|customer_total_spend|
+--------------------+--------------------+
|f82d1df095bf073a1...|              117.57|
|6442504c76c94895e...|               73.34|
|e139948a02a65f3d6...|              140.62|
|fb6f8e0cb2c65dfad...|              155.27|
|8baeca32aac79a831...|               59.87|
|0b60138b038f460ba...|               87.39|
|f39d2cea42fe92f3f...|               63.42|
|18143f5d1df138267...|                65.1|
|eb7b6d66f9d085148...|  45.099999999999994|
|fc021f2dd664b4945...|               55.11|
+--------------------+--------------------+
only showing top 10 rows


In [66]:
customer_summary = customer_order_summary.join(
    customer_spend_summary,
    on="customer_id",
    how="left"
)

customer_summary = customer_summary.withColumn(
    "customer_is_repeat",
    when(col("customer_order_count") > 1, 1).otherwise(0)
)

customer_summary.show(10)

+--------------------+--------------------+--------------------+------------------+
|         customer_id|customer_order_count|customer_total_spend|customer_is_repeat|
+--------------------+--------------------+--------------------+------------------+
|088935be5a63fd416...|                   1|               83.34|                 0|
|1187acd131fe97cfc...|                   1|               46.11|                 0|
|0b60138b038f460ba...|                   1|               87.39|                 0|
|6b75e2df886b0fdde...|                   1|              139.06|                 0|
|fdd48a32f403170cf...|                   1|               57.16|                 0|
|42da09831872a4ecc...|                   1|               88.83|                 0|
|c920352ccf11e554a...|                   1|               75.07|                 0|
|6442504c76c94895e...|                   1|               73.34|                 0|
|e139948a02a65f3d6...|                   1|              140.62|            

In [67]:
order_items = order_items.join(
    customer_summary,
    on="customer_id",
    how="left"
)

print(order_items.count(), len(order_items.columns))

112650 50


In [68]:
seller_summary = order_items.groupBy("seller_id").agg(
    count("*").alias("seller_item_count")
)

seller_summary.show(10)

+--------------------+-----------------+
|           seller_id|seller_item_count|
+--------------------+-----------------+
|ff063b022a9a0aab9...|               96|
|8e6cc767478edae94...|              114|
|a49928bcdf77c55c6...|              106|
|da7039f29f90ce5b4...|               18|
|062ce95fa2ad4dfae...|               60|
|2009a095de2a2a416...|               12|
|0ea22c1cfbdc755f8...|              269|
|6eeed17989b0ae47c...|                3|
|e63e8bfa530fb1691...|               43|
|4d600e08ecbe08258...|                8|
+--------------------+-----------------+
only showing top 10 rows


In [69]:
seller_summary = seller_summary.join(
    order_items.groupBy("seller_id").agg(
        countDistinct("order_id").alias("seller_order_count")
    ),
    on="seller_id",
    how="left"
)

seller_summary.show(10)

+--------------------+-----------------+------------------+
|           seller_id|seller_item_count|seller_order_count|
+--------------------+-----------------+------------------+
|ff063b022a9a0aab9...|               96|                94|
|8e6cc767478edae94...|              114|               106|
|a49928bcdf77c55c6...|              106|                98|
|da7039f29f90ce5b4...|               18|                17|
|062ce95fa2ad4dfae...|               60|                55|
|2009a095de2a2a416...|               12|                 8|
|0ea22c1cfbdc755f8...|              269|               237|
|6eeed17989b0ae47c...|                3|                 3|
|e63e8bfa530fb1691...|               43|                42|
|4d600e08ecbe08258...|                8|                 8|
+--------------------+-----------------+------------------+
only showing top 10 rows


In [70]:
seller_sales = order_items.groupBy("seller_id").agg(
sum("item_total").alias("seller_total_sales")
)

seller_summary = seller_summary.join(
seller_sales,
on="seller_id",
how="left"
)

seller_summary.show(10)


+--------------------+-----------------+------------------+------------------+
|           seller_id|seller_item_count|seller_order_count|seller_total_sales|
+--------------------+-----------------+------------------+------------------+
|ff063b022a9a0aab9...|               96|                94|14061.270000000002|
|8e6cc767478edae94...|              114|               106|           9115.41|
|a49928bcdf77c55c6...|              106|                98|10574.250000000004|
|da7039f29f90ce5b4...|               18|                17|           1503.55|
|062ce95fa2ad4dfae...|               60|                55|           8721.44|
|2009a095de2a2a416...|               12|                 8|            582.43|
|0ea22c1cfbdc755f8...|              269|               237|15075.310000000003|
|6eeed17989b0ae47c...|                3|                 3|            676.01|
|e63e8bfa530fb1691...|               43|                42|1979.1699999999998|
|4d600e08ecbe08258...|                8|            

In [71]:
seller_summary = seller_summary.withColumn(
"seller_avg_order_value",
col("seller_total_sales") / col("seller_order_count")
)

seller_summary.show(10)


+--------------------+-----------------+------------------+------------------+----------------------+
|           seller_id|seller_item_count|seller_order_count|seller_total_sales|seller_avg_order_value|
+--------------------+-----------------+------------------+------------------+----------------------+
|ff063b022a9a0aab9...|               96|                94|14061.270000000002|     149.5879787234043|
|8e6cc767478edae94...|              114|               106|           9115.41|     85.99443396226415|
|a49928bcdf77c55c6...|              106|                98|10574.250000000004|    107.90051020408167|
|da7039f29f90ce5b4...|               18|                17|           1503.55|     88.44411764705882|
|062ce95fa2ad4dfae...|               60|                55|           8721.44|    158.57163636363637|
|2009a095de2a2a416...|               12|                 8|            582.43|              72.80375|
|0ea22c1cfbdc755f8...|              269|               237|15075.310000000003|    

In [72]:
order_items = order_items.join(
    seller_summary,
    on="seller_id",
    how="left"
)

print(order_items.count(), len(order_items.columns))

112650 54


In [73]:
order_items = order_items.withColumn(
"customer_avg_order_value",
col("customer_total_spend") / col("customer_order_count")
)

order_items.select(
"customer_id",
"customer_order_count",
"customer_total_spend",
"customer_avg_order_value"
).show(10)


+--------------------+--------------------+--------------------+------------------------+
|         customer_id|customer_order_count|customer_total_spend|customer_avg_order_value|
+--------------------+--------------------+--------------------+------------------------+
|05b5f7bd9f0d0c31b...|                   1|               18.99|                   18.99|
|58dbd0b2d70206bf4...|                   1|  218.04000000000002|      218.04000000000002|
|7b078f45ddc9d7325...|                   1|               77.47|                   77.47|
|96fd44add323bdcc8...|                   1|              911.65|                  911.65|
|9ed5e522dd9dd85b4...|                   1|              880.75|                  880.75|
|a91d3b41b3f176b1e...|                   1|  234.85999999999999|      234.85999999999999|
|aae651df85accd493...|                   1|              129.86|                  129.86|
|c0d3ff03f73f41d3c...|                   1|               86.86|                   86.86|
|d6c4d0640

In [74]:
order_items = order_items.withColumn(
"freight_percentage",
(col("freight_value") / col("price")) * 100
)

order_items.select(
"price",
"freight_value",
"freight_percentage"
).show(10)


+------+-------------+------------------+
| price|freight_value|freight_percentage|
+------+-------------+------------------+
|  58.9|        13.29|22.563667232597624|
| 239.9|        19.93|  8.30762817840767|
| 199.0|        17.87| 8.979899497487438|
| 12.99|        12.79| 98.46035411855273|
| 199.9|        18.14| 9.074537268634316|
|  21.9|        12.69|57.945205479452056|
|  19.9|        11.85| 59.54773869346733|
| 810.0|        70.75| 8.734567901234568|
|145.95|        11.65| 7.982185680027408|
| 53.99|         11.4|21.115021300240784|
+------+-------------+------------------+
only showing top 10 rows


In [75]:
from pyspark.sql.functions import year

order_items = order_items.withColumn(
"order_year",
year("order_purchase_timestamp")
)

order_items.select(
"order_purchase_timestamp",
"order_year"
).show(10)


+------------------------+----------+
|order_purchase_timestamp|order_year|
+------------------------+----------+
|     2017-09-13 08:59:02|      2017|
|     2017-04-26 10:53:06|      2017|
|     2018-01-14 14:33:31|      2018|
|     2018-08-08 10:00:35|      2018|
|     2017-02-04 13:57:51|      2017|
|     2017-05-15 21:42:34|      2017|
|     2017-12-10 11:53:48|      2017|
|     2018-07-04 12:08:27|      2018|
|     2018-03-19 18:40:33|      2018|
|     2018-07-02 13:59:39|      2018|
+------------------------+----------+
only showing top 10 rows


In [76]:
from pyspark.sql.functions import month

order_items = order_items.withColumn(
"order_month",
month("order_purchase_timestamp")
)

order_items.select(
"order_purchase_timestamp",
"order_month"
).show(10)


+------------------------+-----------+
|order_purchase_timestamp|order_month|
+------------------------+-----------+
|     2017-09-13 08:59:02|          9|
|     2017-04-26 10:53:06|          4|
|     2018-01-14 14:33:31|          1|
|     2018-08-08 10:00:35|          8|
|     2017-02-04 13:57:51|          2|
|     2017-05-15 21:42:34|          5|
|     2017-12-10 11:53:48|         12|
|     2018-07-04 12:08:27|          7|
|     2018-03-19 18:40:33|          3|
|     2018-07-02 13:59:39|          7|
+------------------------+-----------+
only showing top 10 rows


In [77]:
final_columns = [
    "order_id",
    "order_item_id",
    "product_id",
    "seller_id",
    "shipping_limit_date",
    "price",
    "freight_value",
    "item_total",
    "customer_id",
    "order_status",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "product_category_name",
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",
    "seller_zip_code_prefix",
    "seller_city",
    "seller_state",
    "total_payment_value",
    "payment_types",
    "max_payment_installments",
    "payment_count",
    "review_count",
    "review_score_avg",
    "review_score_min",
    "review_score_max",
    "customer_unique_id",
    "customer_zip_code_prefix",
    "customer_city",
    "customer_state",
    "product_category_name_english",
    "order_item_count",
    "order_total_value",
    "order_total_freight",
    "delivery_days",
    "estimated_delivery_days",
    "delivery_delay_days",
    "is_delivered_late",
    "delivery_status",
    "customer_order_count",
    "customer_total_spend",
    "customer_is_repeat",
    "seller_item_count",
    "seller_order_count",
    "seller_total_sales",
    "freight_percentage",
    "customer_avg_order_value",
    "seller_avg_order_value",
    "order_year",
    "order_month"
]

item_analysis_spark = order_items.select(final_columns)

print(item_analysis_spark.count(), len(item_analysis_spark.columns))

112650 58


In [78]:
item_analysis_spark.show(5)

+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+----------------------+--------------------+------------+-------------------+-------------+------------------------+-------------+------------+----------------+----------------+----------------+--------------------+------------------------+--------------------+--------------+-----------------------------+----------------+------------------+-------------------+-------------+-----------------------+-------------------+-----------------+---------------+--------------------+--------------------+------------------+-----------------+----

In [79]:
print(item_analysis_spark.columns == final_columns)
print(len(item_analysis_spark.columns))


True
58


In [80]:
item_analysis_spark_sorted = item_analysis_spark.orderBy(
"order_id",
"order_item_id"
)

item_analysis_spark_sorted.select(
"order_id",
"order_item_id",
"item_total",
"customer_total_spend",
"seller_total_sales"
).show(10, truncate=False)


+--------------------------------+-------------+------------------+--------------------+------------------+
|order_id                        |order_item_id|item_total        |customer_total_spend|seller_total_sales|
+--------------------------------+-------------+------------------+--------------------+------------------+
|00010242fe8c5a6d1ba2dd792cb16214|1            |72.19             |72.19               |15183.640000000003|
|00018f77f2f0320c557190d7a144bdd3|1            |259.83            |259.83              |12071.999999999996|
|000229ec398224ef6ca0657da4fc703e|1            |216.87            |216.87              |3548.9500000000003|
|00024acbcdf0a6daa1e931b038114c75|1            |25.78             |25.78               |1331.87           |
|00042b26cf59d7ce69dfabb4e55b4fd9|1            |218.04000000000002|218.04000000000002  |4267.33           |
|00048cc3ae777c65dbb7d2a0634bc1ea|1            |34.589999999999996|34.589999999999996  |1593.82           |
|00054e8431b9d7675808bcb819f

In [81]:
from pyspark.sql.functions import col, sum, when

null_counts = item_analysis_spark.select([
sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
for c in item_analysis_spark.columns
])

null_counts.show(truncate=False)


+--------+-------------+----------+---------+-------------------+-----+-------------+----------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+----------------------+-----------+------------+-------------------+-------------+------------------------+-------------+------------+----------------+----------------+----------------+------------------+------------------------+-------------+--------------+-----------------------------+----------------+-----------------+-------------------+-------------+-----------------------+-------------------+-----------------+---------------+--------------------+--------------------+------------------+-----------------+------------------+------------------+------------------+------------------

In [82]:
print("Total rows:", item_analysis_spark.count())

print(
"Unique order-item keys:",
item_analysis_spark.select(
"order_id",
"order_item_id"
).distinct().count()
)


Total rows: 112650
Unique order-item keys: 112650


In [83]:

item_analysis_spark.printSchema()



root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- item_total: double (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = t

In [84]:

item_analysis_spark.groupBy(
    "delivery_status"
).count().orderBy(
    "delivery_status"
).show()

item_analysis_spark.groupBy(
    "is_delivered_late"
).count().orderBy(
    "is_delivered_late"
).show()



+---------------+------+
|delivery_status| count|
+---------------+------+
|          Early|101481|
|           Late|  7265|
|  Not Delivered|  2454|
|        On Time|  1450|
+---------------+------+

+-----------------+------+
|is_delivered_late| count|
+-----------------+------+
|                0|105385|
|                1|  7265|
+-----------------+------+



In [85]:
output_path = r"C:\MY ALL\Analytics Study\Project\Ecommerce Data Pipeline\Data\analytics_ready\spark\olist_item_analysis"

item_analysis_spark.write.mode("overwrite").parquet(output_path)

print("Parquet write successful!")

Parquet write successful!


In [86]:
parquet_check = spark.read.parquet(output_path)

print("Rows:", parquet_check.count())
print("Columns:", len(parquet_check.columns))
print("Columns match:", parquet_check.columns == final_columns)

Rows: 112650
Columns: 58
Columns match: True


In [87]:
single_parquet_path = r"C:\MY ALL\Analytics Study\Project\Ecommerce Data Pipeline\Data\analytics_ready\spark\olist_item_analysis_single.parquet"

item_analysis_spark.coalesce(1).write.mode("overwrite").parquet(single_parquet_path)

print("Single Parquet file written successfully!")

Single Parquet file written successfully!


In [88]:
import os

single_path = single_parquet_path

parquet_files = [
    f for f in os.listdir(single_path)
    if f.endswith(".parquet")
]

print("Parquet data files:", len(parquet_files))
print("File:", parquet_files[0] if parquet_files else "None")

Parquet data files: 1
File: part-00000-8105c75a-32fa-4a3f-a859-1f153791345b-c000.snappy.parquet
